# Interactive Tutorial: Understanding $k$-Anonymity

Welcome to this interactive tutorial on **$k$-Anonymity**, a foundational privacy-preserving data publishing technique.

---

## 1. Core Concepts & Definitions

**$k$-Anonymity** guarantees that every individual's record in a published dataset cannot be distinguished from at least **$k - 1$** other individuals in the same dataset.

### Key Attribute Types:
1. **Explicit Identifiers**: Directly identify an individual (e.g., Name, SSN, Email). -> *Remove or mask.*
2. **Quasi-Identifiers (QIs)**: Attributes that can re-identify an individual when combined (e.g., Age, Zip Code, Gender). -> *Generalize or suppress into groups of size $\ge k$.*
3. **Sensitive Attributes**: Private research attributes (e.g., Medical diagnosis, Salary). -> *Retain without linking to explicit identifiers.*


## 2. Setup Environment & Load Sample Data

In [1]:
import pandas as pd
import numpy as np

# Sample raw medical dataset
raw_data = {
    'Name': ['Alice', 'Bob', 'Charlie', 'David', 'Eva', 'Frank', 'Grace', 'Hannah'],
    'Age': [22, 27, 29, 25, 33, 38, 31, 36],
    'ZipCode': [14850, 14853, 14850, 14853, 14850, 14853, 14850, 14853],
    'Diagnosis': ['Heart Disease', 'Flu', 'Cancer', 'Bronchitis', 'Diabetes', 'Flu', 'Heart Disease', 'Cancer']
}

df = pd.DataFrame(raw_data)
print("--- Raw Dataset ---")
df

--- Raw Dataset ---


,Name,Age,ZipCode,Diagnosis
0,Alice,22,14850,Heart Disease
1,Bob,27,14853,Flu
2,Charlie,29,14850,Cancer
3,David,25,14853,Bronchitis
4,Eva,33,14850,Diabetes
5,Frank,38,14853,Flu
6,Grace,31,14850,Heart Disease
7,Hannah,36,14853,Cancer


## 3. Applying Anonymization (Generalization & Suppression)

In [6]:
def anonymize_dataset(data, age_bins, zip_mask_digits):
    df_anon = data.copy()

    # Step 1: Drop Explicit Identifier
    if 'Name' in df_anon.columns:
        df_anon = df_anon.drop(columns=['Name'])

    # Step 2: Generalize Age
    age_labels = [f"{age_bins[i]}-{age_bins[i+1]}" for i in range(len(age_bins)-1)]
    df_anon['Age_Group'] = pd.cut(df_anon['Age'], bins=age_bins, labels=age_labels)

    # Step 3: Mask ZipCode (Fixing the 0-digit masking issue)
    if zip_mask_digits == 0:
        df_anon['Zip_Generalized'] = df_anon['ZipCode'].astype(str)
    else:
        df_anon['Zip_Generalized'] = (
            df_anon['ZipCode'].astype(str).str[:-zip_mask_digits]
            + ('*' * zip_mask_digits)
        )

    return df_anon[['Age_Group', 'Zip_Generalized', 'Diagnosis']]

# Generalize age into 10-year intervals, mask last digit of zip
bins = [20, 30, 40]
df_anonymized = anonymize_dataset(df, age_bins=bins, zip_mask_digits=1)
df_anonymized

,Age_Group,Zip_Generalized,Diagnosis
0,20-30,1485*,Heart Disease
1,20-30,1485*,Flu
2,20-30,1485*,Cancer
3,20-30,1485*,Bronchitis
4,30-40,1485*,Diabetes
5,30-40,1485*,Flu
6,30-40,1485*,Heart Disease
7,30-40,1485*,Cancer


## 4. Calculating Equivalence Classes & Checking $k$

In [7]:
qi_cols = ['Age_Group', 'Zip_Generalized']
eq_classes = df_anonymized.groupby(qi_cols, observed=False).size().reset_index(name='Group_Size')
k_min = eq_classes['Group_Size'].min()

print("Equivalence Classes Breakdown:")
display(eq_classes)
print(f"\nMinimum k-value for dataset: {k_min}")

Equivalence Classes Breakdown:


,Age_Group,Zip_Generalized,Group_Size
0,20-30,1485*,4
1,30-40,1485*,4



Minimum k-value for dataset: 4


## 5. Interactive Sandbox (Test Parameters & Custom Inputs)

Run the cell below to adjust parameters interactively using slider controls.

In [8]:
from ipywidgets import interact, widgets

@interact(
    bin_step=widgets.IntSlider(min=5, max=20, step=5, value=10, description='Age Bin Width:'),
    zip_mask=widgets.IntSlider(min=0, max=3, step=1, value=1, description='Zip Mask Digits:'),
    target_k=widgets.IntSlider(min=1, max=5, step=1, value=2, description='Target k:')
)
def interactive_anonymizer(bin_step, zip_mask, target_k):
    age_bins = list(range(20, 50, bin_step))
    df_anon = anonymize_dataset(df, age_bins=age_bins, zip_mask_digits=zip_mask)

    qi_cols = ['Age_Group', 'Zip_Generalized']
    eq_classes = df_anon.groupby(qi_cols, observed=True).size().reset_index(name='Group_Size')
    actual_k = eq_classes['Group_Size'].min()

    print(f"=== Target k: {target_k} | Achieved k: {actual_k} ===")
    if actual_k >= target_k:
        print(f"✅ PASS: Dataset satisfies {target_k}-anonymity!")
    else:
        print(f"⚠️ FAIL: Dataset violates {target_k}-anonymity (smallest group has {actual_k} records).")

    display(df_anon.head(10))
    print("\nEquivalence Classes:")
    display(eq_classes)

interactive(children=(IntSlider(value=10, description='Age Bin Width:', max=20, min=5, step=5), IntSlider(valu…

## 6. Advanced Extensions: $l$-Diversity & $t$-Closeness

### $l$-Diversity Checker Example:
Ensures every equivalence class has at least $l$ distinct sensitive attribute values.

In [ ]:
# Check distinct sensitive values per equivalence class
l_diversity = df_anonymized.groupby(qi_cols)['Diagnosis'].nunique().reset_index(name='distinct_diagnoses')
min_l = l_diversity['distinct_diagnoses'].min()

print(f"Minimum l-diversity: {min_l}")
display(l_diversity)